In [0]:

import pyspark

In [0]:
# PySpark import
import pyspark

# Spark SQL functions
from pyspark.sql.functions import *

# Column functions
from pyspark.sql.functions import col

# Alias for functions
import pyspark.sql.functions as F

# Spark SQL data types
from pyspark.sql.types import *

In [0]:
# Part A - Loading & preparation

In [0]:
# Load airports.csv and runways.csv

df_airports = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/airport_catalog/airport_schema/airport_volume/airports.csv")

df_runways = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/airport_catalog/airport_schema/airport_volume/runways.csv")

In [0]:
# Display first rows

df_airports.show(5, truncate=False)

df_runways.show(5, truncate=False)

+------+-----+-------------+--------------------+-----------------+------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+------------------------------------------------------------------------------+--------------+--------+
|id    |ident|type         |name                |latitude_deg     |longitude_deg     |elevation_ft|continent|iso_country|iso_region|municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|home_link                                                                     |wikipedia_link|keywords|
+------+-----+-------------+--------------------+-----------------+------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+------------------------------------------------------------------------------+--------------+--------+
|6523  |00A  |heliport     |Total RF Heliport   |40.070985        |-74.933689  

In [0]:
# Cast columns to numeric types

In [0]:


# Airports dataframe

df_airports = df_airports \
    .withColumn("latitude_deg", col("latitude_deg").cast("double")) \
    .withColumn("longitude_deg", col("longitude_deg").cast("double")) \
    .withColumn("elevation_ft", col("elevation_ft").cast("integer"))

In [0]:
# Runways dataframe

df_runways = df_runways \
    .withColumn("length_ft", col("length_ft").cast("integer")) \
    .withColumn("width_ft", col("width_ft").cast("integer"))

In [0]:
# Print schema and row count

# Airports

df_airports.printSchema()

print("Airport Rows:", df_airports.count())

root
 |-- id: integer (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)

Airport Rows: 85324


In [0]:

# Runways

df_runways.printSchema()

print("Runway Rows:", df_runways.count())

root
 |-- id: integer (nullable = true)
 |-- airport_ref: integer (nullable = true)
 |-- airport_ident: string (nullable = true)
 |-- length_ft: integer (nullable = true)
 |-- width_ft: integer (nullable = true)
 |-- surface: string (nullable = true)
 |-- lighted: integer (nullable = true)
 |-- closed: integer (nullable = true)
 |-- le_ident: string (nullable = true)
 |-- le_latitude_deg: double (nullable = true)
 |-- le_longitude_deg: double (nullable = true)
 |-- le_elevation_ft: integer (nullable = true)
 |-- le_heading_degT: double (nullable = true)
 |-- le_displaced_threshold_ft: integer (nullable = true)
 |-- he_ident: string (nullable = true)
 |-- he_latitude_deg: double (nullable = true)
 |-- he_longitude_deg: double (nullable = true)
 |-- he_elevation_ft: integer (nullable = true)
 |-- he_heading_degT: double (nullable = true)
 |-- he_displaced_threshold_ft: integer (nullable = true)

Runway Rows: 47884


In [0]:
# part B  JOIN & TRANSFORMATIONS



In [0]:
# Inner Join

# df_combined = df_airports.join(
#     df_runways,
#     df_airports.ident == df_runways.airport_ident,
#     "inner"
# )


df_combined = df_airports.alias("a").join(
    df_runways.alias("r"),
    col("a.ident") == col("r.airport_ident"),
    "inner"
).select(
    col("a.id").alias("airport_id"),
    col("a.ident"),
    col("a.type"),
    col("a.name"),
    col("a.latitude_deg"),
    col("a.longitude_deg"),
    col("a.elevation_ft"),
    col("a.continent"),
    col("a.iso_country"),
    col("a.iso_region"),
    col("a.municipality"),

    col("r.airport_ref"),
    col("r.airport_ident"),
    col("r.length_ft"),
    col("r.width_ft"),
    col("r.surface"),
    col("r.lighted"),
    col("r.closed"),
    col("r.le_ident"),
    col("r.he_ident")
)


In [0]:
df_combined = df_combined.withColumn(
    "runway_category",
    when(col("length_ft") > 10000, "Long")
    .when((col("length_ft") >= 5000) & (col("length_ft") <= 9999), "Medium")
    .otherwise("Short")
)

In [0]:
# Display joined data
df_combined.show(5, truncate=False)

+------+-----+-------------+--------------------+-----------------+------------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+------------------------------------------------------------------------------+--------------+--------+------+-----------+-------------+---------+--------+-------+-------+------+--------+---------------+----------------+---------------+---------------+-------------------------+--------+---------------+----------------+---------------+---------------+-------------------------+
|id    |ident|type         |name                |latitude_deg     |longitude_deg     |elevation_ft|continent|iso_country|iso_region|municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|home_link                                                                     |wikipedia_link|keywords|id    |airport_ref|airport_ident|length_ft|width_ft|surface|lighted|closed|le_ident|le_latitude_deg|le_longitude

In [0]:
# Add runway_category column


df_combined = df_combined.withColumn(
    "runway_category",
    when(col("length_ft") > 10000, "Long")
    .when((col("length_ft") >= 5000) & (col("length_ft") <= 9999), "Medium")
    .otherwise("Short")
)

In [0]:
# Verify 



df_combined.select(
    "name",
    "length_ft",
    "runway_category"
).show(10, truncate=False)

+-----------------------+---------+---------------+
|name                   |length_ft|runway_category|
+-----------------------+---------+---------------+
|Total RF Heliport      |80       |Short          |
|Lowell Field           |2500     |Short          |
|Epps Airpark           |2100     |Short          |
|Katmai Lodge Airport   |4517     |Short          |
|Fulton Airport         |1450     |Short          |
|Cordes Airport         |1700     |Short          |
|Goldstone (GTS) Airport|6000     |Medium         |
|Grass Patch Airport    |3200     |Short          |
|River Oak Airport      |4700     |Short          |
|Lt World Airport       |2600     |Short          |
+-----------------------+---------+---------------+
only showing top 10 rows


In [0]:
# PART C — SQL ANALYSIS

In [0]:
# Create Temp View

df_combined.createOrReplaceTempView("airports_view")

In [0]:
# Top 10 countries with highest runways

spark.sql("""
SELECT 
    iso_country,
    COUNT(*) AS runway_count
FROM airports_view
GROUP BY iso_country
ORDER BY runway_count DESC
LIMIT 10
""").show()

+-----------+------------+
|iso_country|runway_count|
+-----------+------------+
|         US|       26450|
|         BR|        5825|
|         AU|        2103|
|         CA|        1578|
|         AR|         761|
|         FR|         659|
|         GB|         600|
|         DE|         592|
|         RU|         489|
|         ID|         414|
+-----------+------------+



In [0]:
# Average runway length per continent
spark.sql("""
SELECT 
    continent,
    AVG(length_ft) AS avg_runway_length
FROM airports_view
WHERE length_ft IS NOT NULL
GROUP BY continent
ORDER BY avg_runway_length DESC
""").show()

+---------+------------------+
|continent| avg_runway_length|
+---------+------------------+
|       AN| 7768.857142857143|
|       AS|6925.6469996647675|
|       AF| 6787.526448362721|
|       EU| 4103.151335311572|
|       OC|3558.0401785714284|
|       SA|2857.0891141239963|
|       NA| 2599.724218941758|
+---------+------------------+



In [0]:
# Airports in India with lighted runway
spark.sql("""
SELECT 
    name,
    municipality,
    COUNT(*) AS lighted_runways
FROM airports_view
WHERE iso_country = 'IN'
AND lighted = 1
GROUP BY name, municipality
ORDER BY lighted_runways DESC
""").show()

+--------------------+------------+---------------+
|                name|municipality|lighted_runways|
+--------------------+------------+---------------+
|Indira Gandhi Int...|   New Delhi|              3|
|Navi Mumbai Inter...| Navi Mumbai|              2|
|Raja Bhoj Interna...|      Bhopal|              2|
|Rajiv Gandhi Inte...|   Hyderabad|              2|
|Netaji Subhash Ch...|     Kolkata|              2|
|Agra Airport / Ag...|        Agra|              2|
|   Prayagraj Airport|   Allahabad|              2|
|Ambala Air Force ...|      Ambala|              2|
|Chennai Internati...|     Chennai|              2|
|Kempegowda Intern...|   Bengaluru|              2|
|Mangaluru Interna...|   Mangaluru|              2|
|Bidar Airport / B...|       Bidar|              2|
|Chhatrapati Shiva...|      Mumbai|              2|
|Tambaram Air Forc...|     Chennai|              2|
|       Jammu Airport|       Jammu|              1|
|      Dhulia Airport|        NULL|              1|
|   Shravast

In [0]:
# Top 5 airports with longest runway

spark.sql("""
SELECT 
    name,
    iso_country,
    MAX(length_ft) AS max_runway_length
FROM airports_view
GROUP BY name, iso_country
ORDER BY max_runway_length DESC
LIMIT 5
""").show()

+--------------------+-----------+-----------------+
|                name|iso_country|max_runway_length|
+--------------------+-----------+-----------------+
|Gunflint Seaplane...|         US|            30000|
|Libby Camps Seapl...|         US|            26000|
|Brookville Reserv...|         US|            25000|
|Long Lake Seaplan...|         US|            25000|
|Conchas Lake Seap...|         US|            21120|
+--------------------+-----------+-----------------+



In [0]:
# PART D — SAVE AS PARQUET

In [0]:
# Save parquet file


df_combined.write \
    .mode("overwrite") \
    .parquet("/Volumes/airport_catalog/airport_schema/airport_volume/airport_parquet")

In [0]:
# Read parquet file

df_parquet = spark.read.parquet(
    "/Volumes/airport_catalog/airport_schema/airport_volume/airport_parquet"
)

df_parquet.show(10, truncate=False)

+----------+-----+-------------+-----------------------+------------------+-------------------+------------+---------+-----------+----------+------------+-----------+-------------+---------+--------+-------+-------+------+--------+--------+---------------+
|airport_id|ident|type         |name                   |latitude_deg      |longitude_deg      |elevation_ft|continent|iso_country|iso_region|municipality|airport_ref|airport_ident|length_ft|width_ft|surface|lighted|closed|le_ident|he_ident|runway_category|
+----------+-----+-------------+-----------------------+------------------+-------------------+------------+---------+-----------+----------+------------+-----------+-------------+---------+--------+-------+-------+------+--------+--------+---------------+
|6523      |00A  |heliport     |Total RF Heliport      |40.070985         |-74.933689         |11          |NA       |US         |US-PA     |Bensalem    |6523       |00A          |80       |80      |ASPH-G |1      |0     |H1     

In [0]:
# Verify parquet data


df_parquet.show(10, truncate=False)

+----------+-----+-------------+-----------------------+------------------+-------------------+------------+---------+-----------+----------+------------+-----------+-------------+---------+--------+-------+-------+------+--------+--------+---------------+
|airport_id|ident|type         |name                   |latitude_deg      |longitude_deg      |elevation_ft|continent|iso_country|iso_region|municipality|airport_ref|airport_ident|length_ft|width_ft|surface|lighted|closed|le_ident|he_ident|runway_category|
+----------+-----+-------------+-----------------------+------------------+-------------------+------------+---------+-----------+----------+------------+-----------+-------------+---------+--------+-------+-------+------+--------+--------+---------------+
|6523      |00A  |heliport     |Total RF Heliport      |40.070985         |-74.933689         |11          |NA       |US         |US-PA     |Bensalem    |6523       |00A          |80       |80      |ASPH-G |1      |0     |H1     